As the title suggests, this short script is to find the overlap between homeostatic and kinetic and essential genes. This is to KD the genes and find out whether there would be an effect on growth.

In [75]:
import pandas as pd
import os, dill
import numpy as np
import cvxpy as cp
from typing import Iterable, Optional, Mapping, cast
from plotly import graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

from ecoli.processes.metabolism_redux_classic import NetworkFlowModel, FlowResult

os.chdir(os.path.expanduser('~/dev/vEcoli'))

%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
def load_sim(
        time_num: int,
        date: str,
        experiment_name: str,
        condition: str,
        experiment_type: str,
):
    # --- Load Sim ---
    time = str(time_num)
    entry = f'{experiment_name}_{time}_{date}'
    folder_path = f'out/{experiment_type}/{condition}/{entry}/'

    output = np.load(folder_path + '0_output.npy', allow_pickle='TRUE').item()
    output = output['agents']['0']
    fba = output['listeners']['fba_results']
    bulk = pd.DataFrame(output['bulk'])
    f = open(folder_path + 'agent_steps.pkl', 'rb')
    agent = dill.load(f)
    f.close()

    metabolism = agent['ecoli-metabolism-redux-classic']

    return fba, bulk, metabolism, output

In [9]:
def get_subset_S(S, met_of_interest):
    S_met = S.loc[met_of_interest, :]
    S_met = S_met.loc[:, ~np.all(S_met == 0, axis=0)]
    return S_met, S_met.columns

In [71]:
from scipy.sparse import csr_matrix
import cvxpy as cp


def get_upstream_reactions(met_of_interest, metabolism=metabolism):
    """Run parsimonious FBA (pFBA): maximize metabolite production then minimize total flux.
    Same network and constraints as NetworkFlowModel; only the objective differs.
    Returns reaction names with nonzero flux in the minimal-flux solution."""
    network_flow_model = metabolism.network_flow_model

    reaction_names = metabolism.reaction_names.copy()
    metabolites = metabolism.metabolite_names.copy()
    met_index = metabolites.index(met_of_interest)
    intermediate_idx = network_flow_model.intermediates_idx.copy()

    all_exchanges = metabolism.exchange_molecules
    uptakes = metabolism.allowed_exchange_uptake

    # set up exchanges
    S_exch = np.zeros((len(metabolites), len(all_exchanges) + len(uptakes)))
    exchanges = []
    secretion_idx = []
    exch_idx = 0
    for met in all_exchanges:
        exch_name = met + " exchange"
        met_idx = network_flow_model.met_map[met]
        if met in uptakes:
            S_exch[met_idx, exch_idx] = 1
            exchanges.append(exch_name)
            exch_idx += 1
        exchanges.append(exch_name + " rev")
        secretion_idx.append(exch_idx)
        S_exch[met_idx, exch_idx] = -1
        exch_idx += 1

    S_exch = csr_matrix(S_exch)

    _, n_exch_rxns = S_exch.shape

    v = cp.Variable(len(reaction_names), nonneg=True)
    e = cp.Variable(n_exch_rxns, nonneg=True)
    dm = network_flow_model.S_orig @ v + S_exch @ e
    upper_flux_bound = 100
    try:
        temp = np.where(intermediate_idx == met_index)[0]
        np.delete(intermediate_idx, temp)
    except:
        print(f'Metabolite {met_of_interest} not found in intermediates list')

    # same constraints as NetworkFlowModel
    constr = [
        dm[intermediate_idx] == 0,
        v >= 0,
        v <= upper_flux_bound,
        e >= 0,
        e <= upper_flux_bound,
        cp.sum(v) >= 10,  # avoid trivial zero-flux solution
    ]
    if network_flow_model.maintenance_idx is not None:
        constr.append(v[network_flow_model.maintenance_idx] == 0)

    # Maximize production/export of the metabolite while minimizing total flux
    prob = cp.Problem(cp.Minimize(-dm[met_index] + cp.sum(v)), constr)
    prob.solve(solver=cp.GLOP)
    if prob.status != "optimal":
        raise ValueError(f"pFBA did not solve: {prob.status}")

    solution = pd.DataFrame(v.value, index=reaction_names)
    reactions_making_met = solution.loc[solution[0] != 0].index
    return reactions_making_met


In [24]:
def metabolite_to_catalyst(met_of_interest, metabolism=metabolism):
    '''Given a metabolite, this function first maps the metabolite to the reactions it participates in,
    then maps the reactions to the enzymes that catalyze them, and finally returns a list of unique enzymes
    that are involved in the metabolite's metabolism.'''
    # --- load common variables ---
    metabolites = metabolism.metabolite_names.copy()
    reaction_names = metabolism.reaction_names.copy()
    reaction_catalysts = metabolism.parameters['reaction_catalysts'].copy()

    S = metabolism.stoichiometry.copy()
    S = pd.DataFrame(S, index=metabolites, columns=reaction_names)

    # --- metabolite to reactions ---
    met_to_reactions = dict()
    met_to_catalysts = dict()
    for met in met_of_interest:
        if met not in metabolites:
            raise ValueError(f'Metabolite {met} not found in the model.')
        _, metabolite_reactions = get_subset_S(S, met_of_interest)
        met_to_reactions[met] = metabolite_reactions.tolist()
        catalysts = set()
        for rxn in metabolite_reactions:
            if rxn not in reaction_names:
                raise ValueError(f'Reaction {rxn} not found in the model.')
            # --- reactions to catalysts ---
            catalysts.update(reaction_catalysts[rxn])
        met_to_catalysts[met] = catalysts
    return met_to_catalysts

In [77]:
# import sim
time_num = 600
date = '2026-03-10'
experiment_name = 'all_shrunk_row304'
condition = 'basal'
experiment_type = 'objective_weight'

fba, bulk, metabolism, output = load_sim(time_num, date, experiment_name, condition, experiment_type)

In [1]:
# import essential complexes
complex_ids_txt = pd.read_csv(
    'notebooks/Heena notebooks/Riley New Genes/proteome_fraction_scatterplot_below_line_essential_complex_ids_16.txt',
    header=None)
complex_ids = np.unique(complex_ids_txt[0].tolist())

NameError: name 'pd' is not defined

In [74]:
# find essential enzymatic complexes
kinetic_enzymes = metabolism.parameters['kinetic_constraint_enzymes']
essential_kinetic_catalysts = np.intersect1d(kinetic_enzymes, complex_ids)
len(essential_kinetic_catalysts)

54

## Adapt 20260209_FBA_to_WC to pseudo-KD the essential kinetic enzymes and find the effect on homeostatic term

In [81]:
stoichiometry = metabolism.stoichiometry.copy()
reaction_names = metabolism.reaction_names
metabolites = metabolism.metabolite_names.copy()
kinetic_reaction_ids = metabolism.kinetic_constraint_reactions.copy()
counts_to_molar = output['listeners']['enzyme_kinetics']['counts_to_molar']

homeostatic_dm_targets = pd.DataFrame(fba['target_homeostatic_dmdt'],
                                      columns=metabolism.homeostatic_metabolites).mul(counts_to_molar, axis=0).mean(
    axis=0)  # in conc
homeostatic_metabolite_counts = pd.DataFrame(fba['homeostatic_metabolite_counts'],
                                             columns=metabolism.homeostatic_metabolites).mul(counts_to_molar,
                                                                                             axis=0).mean(
    axis=0)  # in conc
binary_kinetic_idx = fba['binary_kinetic_idx'][1:]
maintenance = pd.DataFrame(fba["maintenance_target"][1:],
                           columns=['maintenance_reaction']).mul(counts_to_molar[1:], axis=0).mean(axis=0)
kinetic = pd.DataFrame(fba["target_kinetic_fluxes"],
                       columns=metabolism.kinetic_constraint_reactions).mul(counts_to_molar, axis=0).mean(axis=0)

In [82]:
FREE_RXNS = [
    "TRANS-RXN-145",
    "TRANS-RXN0-545",
    "TRANS-RXN0-474",
    "ATPSYN-RXN (reverse)",
]


def test_NetworkFlowModel(weights,
                          modify_kinetic=None, solver_choice=cp.GLOP,
                          homeostatic_metabolite_counts=homeostatic_metabolite_counts,
                          homeostatic_dm_targets=homeostatic_dm_targets,
                          kinetic=kinetic,
                          binary_kinetics_idx=None,
                          maintenance=maintenance,
                          ):
    if modify_kinetic is not None:
        # modify kinetic need to be a dict
        assert isinstance(modify_kinetic, dict), "modify_kinetic should be a dict of reaction: new_flux"
        for rxn, new_flux in modify_kinetic.items():
            if rxn not in kinetic_reaction_ids:
                kinetic_reaction_ids.append(rxn)
                kinetic[rxn] = new_flux
            if rxn in kinetic_reaction_ids:
                kinetic[rxn] = new_flux

    model = NetworkFlowModel(
        stoich_arr=stoichiometry,
        metabolites=metabolites,
        reactions=reaction_names,
        homeostatic_metabolites=metabolism.homeostatic_metabolites,
        kinetic_reactions=metabolism.kinetic_constraint_reactions,
        free_reactions=FREE_RXNS)
    model.set_up_exchanges(exchanges=metabolism.exchange_molecules, uptakes=metabolism.allowed_exchange_uptake)
    solution: FlowResult = model.solve(
        homeostatic_concs=homeostatic_metabolite_counts,  # in conc
        homeostatic_dm_targets=np.array(list(dict(homeostatic_dm_targets).values())),  # conc
        maintenance_target=maintenance,  # conc range
        kinetic_targets=np.array(list(dict(kinetic).values())),  # conc range
        binary_kinetic_idx=binary_kinetics_idx,
        # force_flow_idx=force_reaction_idx,
        objective_weights=weights,  #same
        upper_flux_bound=100,  # increase to 10^9 because notebook runs FlowResult using Counts, WC runs using conc.
        solver=solver_choice)  #SCS. ECOS, MOSEK
    return solution.objective, solution.homeostatic_term, solution.kinetics_term, solution.velocities, solution.dm_dt

In [72]:
get_upstream_reactions(homeostatic_metabolites[0], metabolism)

Index(['DHBDEHYD-RXN', 'ISOCHORMAT-RXN', 'ISOCHORSYN-RXN__ENTC-MONOMER',
       'UDPNACETYLGLUCOSAMENOLPYRTRANS-RXN',
       'UDPNACETYLMURAMATEDEHYDROG-RXN (reverse)'],
      dtype='object')

In [42]:
homeostatic_metabolites[1]

np.str_('2-KETOGLUTARATE[c]')

In [30]:
reaction_catalyst_counts_CPD12261

,DHBDEHYD-RXN,ISOCHORMAT-RXN,ISOCHORSYN-RXN__ENTC-MONOMER,TRANS-RXN0-612-L-PENICILLAMINE//L-PENICILLAMINE.33.,TRANS-RXN0-612-L-PENICILLAMINE//L-PENICILLAMINE.33. (reverse),TRANS-RXN0-612-L-PIPECOLATE//L-PIPECOLATE.27.,TRANS-RXN0-612-L-PIPECOLATE//L-PIPECOLATE.27. (reverse),TRANS-RXN0-612-L-SELENOCYSTEINE//L-SELENOCYSTEINE.35.,TRANS-RXN0-612-L-SELENOCYSTEINE//L-SELENOCYSTEINE.35. (reverse),TRANS-RXN0-612-L-THREO-3-METHYL-ASPARTATE//L-THREO-3-METHYL-ASPARTATE.55.,...,UHPC-RXN,UHPC-RXN (reverse),URIDINEKIN-RXN,URIDYLREM-RXN,URITRANS-RXN,URKI-RXN,glycogen-monomer-extension,DISULFOXRED-RXN[CCO-PERI-BAC]-MONOMER0-4152/MONOMER0-4438//MONOMER0-4438/MONOMER0-4152.71.DEPHOSICITDEHASE-RXN,RXN0-7008-PRO/UBIQUINONE-8//L-DELTA1-PYRROLINE_5-CARBOXYLATE/CPD-9956/PROTON.67.,maintenance_reaction
0,430,-1,1540,20133,20133,20133,20133,20133,20133,20133,...,-1,-1,290,77,-1,290,-1,-1,-1,-1
1,431,-1,1540,20140,20140,20140,20140,20140,20140,20140,...,-1,-1,290,77,-1,290,-1,-1,-1,-1
2,431,-1,1540,20143,20143,20143,20143,20143,20143,20143,...,-1,-1,291,77,-1,291,-1,-1,-1,-1
3,431,-1,1542,20148,20148,20148,20148,20148,20148,20148,...,-1,-1,291,77,-1,291,-1,-1,-1,-1
4,432,-1,1544,20153,20153,20153,20153,20153,20153,20153,...,-1,-1,291,77,-1,291,-1,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,472,-1,1687,23821,23821,23821,23821,23821,23821,23821,...,-1,-1,384,76,-1,384,-1,-1,-1,-1
596,472,-1,1687,23829,23829,23829,23829,23829,23829,23829,...,-1,-1,385,76,-1,385,-1,-1,-1,-1
597,472,-1,1687,23835,23835,23835,23835,23835,23835,23835,...,-1,-1,386,76,-1,386,-1,-1,-1,-1
598,472,-1,1687,23841,23841,23841,23841,23841,23841,23841,...,-1,-1,385,76,-1,385,-1,-1,-1,-1


In [27]:
S = metabolism.stoichiometry.copy()
S = pd.DataFrame(S, index=metabolism.metabolite_names.copy(), columns=
metabolism.reaction_names.copy())

1-2-Diglycerides[c]                         0
1-AMINO-PROPAN-2-ONE-3-PHOSPHATE[c]         0
1-Cys-Peroxiredoxin-L-cysteine[c]           0
1-Cys-Peroxiredoxin-L-hydroxycysteine[c]    0
1-KETO-2-METHYLVALERATE[c]                  0
                                           ..
tRNAs-with-queuine[c]                       0
tRNAs[c]                                    0
trans-delta2-arachidoyl-ACPs[c]             0
type-IV-prepillin[c]                        0
tyrosylated-ssDNA[c]                        0
Name: maintenance_reaction, Length: 6112, dtype: int8